# Creating Spark Session

In [24]:
%pip install beautifulsoup4 textacy nltk

Note: you may need to restart the kernel to use updated packages.


In [25]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.config("spark.sql.session.timeZone", "UTC").appName("FinSentAnalysis").getOrCreate()

# Get Data

### Get News Data

In [0]:
# global constants
API_KEY : str = ''
BEZINGA_URL : str = 'https://api.benzinga.com/api/v2/news'
STOCK_TICKER : str = 'AAPL'
HEADERS = {"accept": "application/json"}
PAGE_SIZE = 100
PAGE_LIMIT = 400

params = {
    'token': API_KEY,
    'displayOutput' : 'full',
    'pageSize' : PAGE_SIZE,
    "sort": "created:asc",
    'tickers': STOCK_TICKER,
    'channels' : 'news'
}

In [0]:
import requests

news_data = []

params['dateFrom'] = '2015-01-01'
params['dateTo'] = '2020-12-31'

for page in range(PAGE_LIMIT + 1):
    params['page'] = page
    response = requests.get(BEZINGA_URL, headers=HEADERS, params=params)
    news_data += response.json()

params['dateFrom'] = '2021-01-01'
params['dateTo'] = '2023-12-31'

for page in range(PAGE_LIMIT + 1):
    params['page'] = page
    response = requests.get(BEZINGA_URL, headers=HEADERS, params=params)
    news_data += response.json()

params['dateFrom'] = '2024-01-01'
params['dateTo'] = '2025-12-31'

for page in range(PAGE_LIMIT + 1):
    params['page'] = page
    response = requests.get(BEZINGA_URL, headers=HEADERS, params=params)
    news_data += response.json()


In [0]:
import json

# Define the filename
dbfs_path = "dbfs:/Volumes/workspace/default/ensf612/aapl_news.json"

dbutils.fs.put(dbfs_path, json.dumps(news_data), overwrite=True)

### Get Stock Data

In [0]:
# global constants
API_KEY : str = ''
API_SECRET_KEY : str = ''
STOCK_TICKER : str = 'AAPL'
ALPACA_URL : str = f'https://data.alpaca.markets/v2/stocks/{STOCK_TICKER}/bars'
HEADERS = {
    "accept": "application/json",
    "APCA-API-KEY-ID": API_KEY,
    "APCA-API-SECRET-KEY": API_SECRET_KEY
    }

params = {
    'timeframe' : '1D',
    'limit' : 10000,
    'adjustment' : 'all'
}

In [0]:
import requests

stock_data = []

params['start'] = '2015-01-01'
params['end'] = '2018-12-31'

response = requests.get(ALPACA_URL, headers=HEADERS, params =params)
stock1 = response.json()['bars']
stock_data += stock1

params['start'] = '2019-01-01'
params['end'] = '2021-12-31'

response = requests.get(ALPACA_URL, headers=HEADERS, params =params)
stock2 = response.json()['bars']
stock_data += stock2

params['start'] = '2022-01-01'
params['end'] = '2025-11-05'

response = requests.get(ALPACA_URL, headers=HEADERS, params =params)
stock3 = response.json()['bars']
stock_data += stock3

In [0]:
import json

# Define the filename
dbfs_path = "dbfs:/Volumes/workspace/default/ensf612/aapl_price.json"

dbutils.fs.put(dbfs_path, json.dumps(stock_data), overwrite=True)

---

# Cataloging Data

In [ ]:
import pandas as pd

workspace = 'danish.shahid@ucalgary.ca'

news_df = pd.read_json(f"../data/aapl_news.json")
price_df = pd.read_json(f"../data/aapl_price.json")

sp_news_df = spark.createDataFrame(news_df)
sp_price_df = spark.createDataFrame(price_df)

sp_news_df.write.json("dbfs:/Volumes/workspace/default/ensf612/aapl_news.json", mode="overwrite")
sp_price_df.write.json("dbfs:/Volumes/workspace/default/ensf612/aapl_price.json", mode="overwrite")

# Reading Data

In [26]:
news_df = spark.read.json("dbfs:/Volumes/workspace/default/ensf612/aapl_news.json")
price_df = spark.read.json("dbfs:/Volumes/workspace/default/ensf612/aapl_price.json")

### Fixing time stamps

In [27]:
news_df = news_df.withColumn('created', regexp_replace('created', r"^[A-Za-z]{3},\s+", "")).withColumn('created', to_timestamp('created', "dd MMM yyyy HH:mm:ss Z")).orderBy('created')
price_df = price_df.withColumn('t', to_timestamp('t', "yyyy-MM-ddTHH:mm:ssZ")).orderBy('t')

### Removing HTML tagging

In [28]:
from bs4 import BeautifulSoup

@udf
def parseHTML(text):
  return BeautifulSoup(text, "html.parser").get_text()

news_df = news_df.withColumn('body', parseHTML(col('body'))).withColumn('title', parseHTML(col('title')))

### Removing New Line Characters

In [29]:
news_df = news_df.withColumn('body', regexp_replace('body', r'\r|\n|\t', ' ')).withColumn('title', regexp_replace('title', r'\r|\n|\t', ' '))

In [30]:
from textacy.preprocessing import *

@udf
def removeUrls(text):
  return replace.urls(text)

news_df = news_df.withColumn('body', removeUrls('body')).withColumn('title', removeUrls('title'))

In [31]:
display(news_df.limit(10))

PythonException: 
  An exception was thrown from the Python worker. Please see the stack trace below.
Traceback (most recent call last):
  File "/databricks/spark/python/_engine_pyspark.zip/_engine_pyspark/serializers.py", line 194, in _read_with_length
    return self.loads(obj)
           ^^^^^^^^^^^^^^^
  File "/databricks/spark/python/_engine_pyspark.zip/_engine_pyspark/serializers.py", line 654, in loads
    return cloudpickle.loads(obj, encoding=encoding)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/databricks/python/lib/python3.12/site-packages/pyspark/cloudpickle/cloudpickle.py", line 475, in subimport
    __import__(name)
ModuleNotFoundError: No module named 'textacy'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/databricks/spark/python/_engine_pyspark.zip/_engine_pyspark/worker.py", line 2699, in main
    func, profiler, deserializer, serializer = read_udfs(pickleSer, infile, eval_type)
     

DataFrame[author: string, body: string, channels: array<struct<name:string>>, created: timestamp, id: bigint, image: array<struct<size:string,url:string>>, stocks: array<struct<exchange:string,name:string>>, tags: array<struct<name:string>>, teaser: string, title: string, updated: string, url: string]